In [1]:
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [2]:
from rasterio.crs import CRS

In [3]:
import os

In [4]:
import netCDF4
import rioxarray
import rasterio

In [5]:
base = os.path.join(os.getcwd(),'..')

In [7]:
data = os.path.join(base,'data(LPJmL)','simulation_global_gwwd2_pushkar')

In [8]:
for r,d,f in os.walk(data):
    for fl in f:
        if fl.endswith('.nc') and 'cft_evap' in fl:
            print(fl)
            fn = os.path.join(r,fl)


cft_evap.nc


In [9]:
ls = ("temperate_cereals","rice","maize","tropical cereals","pulses","temperate roots",
             "tropical roots","sunflower","soybeans","groundnuts","rapeseed","sugarcane", 
      "barley","cotton","wheat2","rice2", "rice3","others","grasses","biofuels1","biofuels2")


In [10]:
# fn = os.path.join(r,'cft_evap.nc')
ds = netCDF4.Dataset(fn)
# nc_file = fn
# ds = xr.open_dataset(nc_file, decode_times=False)

In [11]:
# import xarray as xr
# import rasterio
# from rasterio.transform import from_origin
# import numpy as np
# from pathlib import Path


# nc_file = fn
# out_dir = Path("tiff_output")
# out_dir.mkdir(exist_ok=True)


# ds = xr.open_dataset(nc_file)

# evap = ds["evap"]                 # (time, pft, lat, lon)
# lat = ds["lat"].values
# lon = ds["lon"].values
# pft_names = ds["NamePFT"].values  # strings

# dlon = lon[1] - lon[0]
# dlat = lat[1] - lat[0]

# da = da.squeeze()
# da = da.sortby("lat", ascending=False)

# transform = from_origin(
#     lon.min()-dlon/2,
#     lat.max()+dlat/2,
#     dlon,
#     dlat
# )

# # evap.sizes["time"]
# for t in range(1):
#     time_val = ds["time"].values[t]

#     out_file = out_dir / f"evap_time_{t:03d}.tif"

#     # Data shape: (pft, lat, lon)
#     data = evap.isel(time=t).values.astype(np.float32)

#     with rasterio.open(
#         out_file,
#         "w",
#         driver="GTiff",
#         height=lat,
#         width=lon,
#         count=data.shape[0],      # number of PFTs
#         dtype="float32",
#         crs="""GEOGCS["WGS 84",
#         DATUM["WGS_1984",
#             SPHEROID["WGS 84",6378137,298.257223563]],
#         PRIMEM["Greenwich",0],
#         UNIT["degree",0.0174532925199433]]""",
#         transform=transform,
#         compress="deflate",
#         tiled=True,
#         blockxsize=256,
#         blockysize=256,
#     ) as dst:

#         for i, pft_name in enumerate(pft_names):
#             band = data[i, :, :]
#             dst.write(band, i + 1)
#             dst.set_band_description(i + 1, str(pft_name))

#     print(f"Written {out_file}")


In [36]:
# fn = os.path.join(r,'cft_evap.nc')
nc_file = fn
var_name = "evap"          
os.makedirs(os.path.join(base,'tiffs',var_name),exist_ok = True)
out_tif = os.path.join(base,'tiffs',var_name,var_name+'.tiff')
nodata = -9999.0

ds = xr.open_dataset(nc_file, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds["evap"]    
pft_names = ds["NamePFT"].values  # strings

ntime, bands, nlat, nlon = da.shape

lat = da["lat"].values
lon = da["lon"].values

da = da.squeeze()
da = da.sortby("lat", ascending=True)

dlat = abs(lat[1] - lat[0])
dlon = abs(lon[1] - lon[0])


transform = from_origin(
    lon.min()-dlon/2,
    lat.max()+dlat/2,
    dlon,
    dlat
)


fill_value = da.attrs.get("_FillValue", nodata)


for t in range(evap.sizes["time"]):
    time_val = ds["time"].values[t]

    # out_file = out_dir / f"evap_time_{t:03d}.tif"
    out_tif = os.path.join(base,'tiffs',var_name,f"evap_time_{t:03d}.tif")
    # Data shape: (pft, lat, lon)
    data = evap.isel(time=t).values.astype(np.float32)

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=nlat,
        width=nlon,
        count=bands,               
        dtype="float32",
        crs="""GEOGCS["WGS 84",
            DATUM["WGS_1984",
                SPHEROID["WGS 84",6378137,298.257223563]],
            PRIMEM["Greenwich",0],
            UNIT["degree",0.0174532925199433]]""",
        transform=transform,
        nodata=nodata,
        compress="lzw"
    ) as dst:
    
    #     for i in range(bands):
    #         band = da.isel(pft=i).values.astype(np.float32)
    #         band[band == fill_value] = nodata
    #         band = np.nan_to_num(band, nan=nodata)
    #         dst.write(band, i + 1)
    
    # print(f"Saved 42-band GeoTIFF → {out_tif}")

    
        for i, pft_name in enumerate(pft_names):
            band = data[i, :, :]
            dst.write(band, i + 1)
            dst.set_band_description(i + 1, str(pft_name))
            
    with rasterio.open(out_tif) as src:
        data = src.read()      
        profile = src.profile

    data_flipped = np.flip(data, axis=1)
    
    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(data_flipped)


    print(f"Written {out_tif}")


Written notebooks\..\tiffs\evap\evap_time_000.tif
Written notebooks\..\tiffs\evap\evap_time_001.tif
Written notebooks\..\tiffs\evap\evap_time_002.tif
Written notebooks\..\tiffs\evap\evap_time_003.tif
Written notebooks\..\tiffs\evap\evap_time_004.tif
Written notebooks\..\tiffs\evap\evap_time_005.tif
Written notebooks\..\tiffs\evap\evap_time_006.tif
Written notebooks\..\tiffs\evap\evap_time_007.tif
Written notebooks\..\tiffs\evap\evap_time_008.tif
Written notebooks\..\tiffs\evap\evap_time_009.tif
Written notebooks\..\tiffs\evap\evap_time_010.tif
Written notebooks\..\tiffs\evap\evap_time_011.tif
Written notebooks\..\tiffs\evap\evap_time_012.tif
Written notebooks\..\tiffs\evap\evap_time_013.tif
Written notebooks\..\tiffs\evap\evap_time_014.tif
Written notebooks\..\tiffs\evap\evap_time_015.tif
Written notebooks\..\tiffs\evap\evap_time_016.tif
Written notebooks\..\tiffs\evap\evap_time_017.tif
Written notebooks\..\tiffs\evap\evap_time_018.tif
Written notebooks\..\tiffs\evap\evap_time_019.tif


In [31]:
import rasterio
import numpy as np



In [ ]:
# import rasterio
# from rasterio.transform import from_origin
# import numpy as np

# var_name = "evap"
# nodata = -9999.0

# # Select variable + PFT
# da = ds[var_name].isel(pft=pft_index)

# # Remove size-1 dimensions
# da = da.squeeze(drop=True)

# # 🔴 Fix inverted image
# da = da.sortby("lat", ascending=False)

# ntime, nlat, nlon = da.shape

# lat = da.lat.values
# lon = da.lon.values

# dlat = abs(lat[1] - lat[0])
# dlon = abs(lon[1] - lon[0])

# transform = from_origin(
#     lon.min() - dlon / 2,
#     lat.max() + dlat / 2,
#     dlon,
#     dlat
# )

# # CRS without EPSG dependency
# WGS84_WKT = (
#     'GEOGCS["WGS 84",'
#     'DATUM["WGS_1984",'
#     'SPHEROID["WGS 84",6378137,298.257223563]],'
#     'PRIMEM["Greenwich",0],'
#     'UNIT["degree",0.0174532925199433]]'
# )

# fill_value = da.attrs.get("_FillValue", nodata)

# for t in range(ntime):
#     out_tif = f"evap_{target_pft}_time{t}.tif"

#     band = da.isel(time=t).values.astype(np.float32)
#     band[band == fill_value] = nodata
#     band = np.nan_to_num(band, nan=nodata)

#     with rasterio.open(
#         out_tif,
#         "w",
#         driver="GTiff",
#         height=nlat,
#         width=nlon,
#         count=1,
#         dtype="float32",
#         crs=WGS84_WKT,
#         transform=transform,
#         nodata=nodata,
#         compress="lzw"
#     ) as dst:
#         dst.write(band, 1)

#     print("Written:", out_tif)
